<a class="anchor" id="0"></a>
# **Mubileum Duration Generation with VAE**

<a class="anchor" id="0.1"></a>
# **Table of Contents**

- 1. [Setup](#1)
- 2. [Data Preparation](#2)
<!--     - 2.1 [Data Preparation for LSTM](#2.1) -->
- 3. [EDA](#3)
- 4. [Anomaly Detection](#4)
- 5. [Anomaly Creation](#5)
- 6. [Genetic Algoritm Otimization](#6)


# **1. SetUp** <a class="anchor" id="1"></a>

[Table of Contents](#0.1)

In [1]:
# Data Manipulation and Analysis
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
import os
from scipy.stats import norm

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn for Machine Learning
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay


import pickle
from sklearn.metrics import precision_recall_curve


# Read a CSV file into a Pandas DataFrame
df = pd.read_csv("/kaggle/input/mobileum/df_anomaly.csv")

In [2]:
filters = pd.read_csv("/kaggle/input/mobileum-empty-agg/not_empty_combinations.csv")

In [3]:
filters

,Call Type,Event Type,Subscriber Type,Traffic Type,Service Type,Partner Description,Number of examples (rows)
0,SMS,ORIGINATED,Postpaid,OFF-NET,NOT-ROAMING,P_0001,2110
1,SMS,ORIGINATED,Postpaid,INT,NOT-ROAMING,P_0001,2110
2,SMS,ORIGINATED,Postpaid,ON-NET,NOT-ROAMING,P_0001,2110
3,SMS,ORIGINATED,Roamer In,OFF-NET,NOT-ROAMING,P_0043,4
4,SMS,ORIGINATED,Roamer In,OFF-NET,NOT-ROAMING,P_0044,7
...,...,...,...,...,...,...,...
1550,VOICE,ROAMING OUT,Prepaid,INT,ROAMING,P_0001,217
1551,VOICE,ROAMING OUT,Prepaid,ON-NET,ROAMING,P_0001,2110
1552,VOICE,ROAMING OUT,Other,OFF-NET,ROAMING,P_0001,2110
1553,VOICE,ROAMING OUT,Other,INT,ROAMING,P_0001,2110


In [4]:
df

,Call Type,Event Type,Subscriber Type,Traffic Type,Service Type,Partner Description,#Events,TIMESTAMP,Anomaly
0,SMS,ORIGINATED,Postpaid,OFF-NET,NOT-ROAMING,P_0001,105,2022-07-18 00:00:00,0
1,VOICE,CALL FORWARD,Application,INT,NOT-ROAMING,P_0001,16,2022-07-18 00:00:00,0
2,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0002,7,2022-07-18 00:00:00,0
3,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0003,15,2022-07-18 00:00:00,0
4,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0004,1,2022-07-18 00:00:00,0
...,...,...,...,...,...,...,...,...,...
958475,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0280,30,2022-10-13 21:00:00,0
958476,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0079,265,2022-10-13 21:00:00,0
958477,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0102,6,2022-10-13 21:00:00,0
958478,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0022,3,2022-10-13 21:00:00,0


# **4. Anomaly Creation** <a class="anchor" id="4"></a>

[Table of Contents](#0.1)

In [5]:
def contain_condition(df, conditions):
    if not conditions:
        return False
    pd_conditions = " & ".join([f'`{key}` == "{value}"' for key, value in conditions.items()])
    return df.eval(pd_conditions).any()

In [6]:
def filter_and_aggregate(df, filter_conditions, hierarchy=False):
    """
    Filters the DataFrame based on given conditions and aggregates by summing up a specified column.
    
    Args:
    - df: The original DataFrame.
    - filter_conditions: A dictionary where keys are column names and values are the values to filter on.
    
    Returns:
    - A new DataFrame with the filtered and aggregated data.
    """
    # Apply the filter conditions
    df_fil = df.copy()
    
    for col, val in filter_conditions.items():
        df_fil = df_fil[df_fil[col] == val]
    
    # Drop the columns used for filtering
    df_fil.drop(columns=list(filter_conditions.keys()), inplace=True)
    
    # Group by 'TIMESTAMP' and sum up the '#Events' column
    if hierarchy:
        df_agg = df_fil.groupby('TIMESTAMP')[['#Events','Anomaly']].sum().reset_index()
    else:
        df_agg = df_fil.groupby('TIMESTAMP')['#Events'].sum().reset_index()
    
    return df_agg


1. get the anomaly_value and the ts with the function. 
2. go to the original df get all the rows with the same ts (TIMESTAMP)
3. from all the row calculate the sum of column (#Events)
4. calculate the multiplier witch m = anomaly_value/ sum of column (#Events)
5. use the multiplier to all rows of the #Events and do the round to get int
6. Create an new columns witch name "Anomaly" with 0 or 1, which all row which has column 
7. gerate an orignal and modified df

The warning message you’ve encountered, "RuntimeWarning: divide by zero encountered in scalar divide," suggests that at some point in your code, there is a division operation where the divisor (sum_events) is zero.

# 5. Hierarchical Structure

In [7]:
df

,Call Type,Event Type,Subscriber Type,Traffic Type,Service Type,Partner Description,#Events,TIMESTAMP,Anomaly
0,SMS,ORIGINATED,Postpaid,OFF-NET,NOT-ROAMING,P_0001,105,2022-07-18 00:00:00,0
1,VOICE,CALL FORWARD,Application,INT,NOT-ROAMING,P_0001,16,2022-07-18 00:00:00,0
2,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0002,7,2022-07-18 00:00:00,0
3,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0003,15,2022-07-18 00:00:00,0
4,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0004,1,2022-07-18 00:00:00,0
...,...,...,...,...,...,...,...,...,...
958475,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0280,30,2022-10-13 21:00:00,0
958476,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0079,265,2022-10-13 21:00:00,0
958477,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0102,6,2022-10-13 21:00:00,0
958478,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0022,3,2022-10-13 21:00:00,0


In [8]:
available_categories = {
    'Call Type': df['Call Type'].unique().tolist(),
    'Event Type': df['Event Type'].unique().tolist(),
    'Subscriber Type': df['Subscriber Type'].unique().tolist(),
    'Traffic Type': df['Traffic Type'].unique().tolist(),
    'Service Type': df['Service Type'].unique().tolist(),
    'Partner Description': df['Partner Description'].unique().tolist(),
    'STOP': False
}

In [9]:
def revert_key_value_mapping(available_categories):
    reversed_mapping = {}
    for key, values in available_categories.items():
        if isinstance(values, list):  # Ensure the value is iterable
            for value in values:
                if value not in reversed_mapping:
                    reversed_mapping[value] = []
                reversed_mapping[value].append(key)
        else:
            # Handle non-list values (like the 'STOP' flag)
            value = values
            if value not in reversed_mapping:
                reversed_mapping[value] = []
            reversed_mapping[value].append(key)
    return reversed_mapping

In [10]:
def create_hierarchy(available_categories):

    hierarchy = {}
    current_column = random.choice(list(available_categories.keys()))
    values = available_categories.get(current_column)
    
    if values:
        for value in values:
            key=value
            new_categories = available_categories.copy()
            new_categories.pop(current_column)
            hierarchy[key] = create_hierarchy(new_categories)
            
    return hierarchy

In [11]:
def flatten_dict_structure(d, rt_available_categories, parent_path=[]):
    items = []
    for k, v in d.items():
        # Determine the category list for the current key from the mapping
        categories = rt_available_categories.get(k)
        
        # Iterate through categories (there could be more than one, but typically one for this use case)
        for category in categories:
            current_path = parent_path + [(category, k)]
            
            if isinstance(v, dict) and v:  # If v is a non-empty dictionary, continue recursion
                items.extend(flatten_dict_structure(v, rt_available_categories, current_path))
            elif not v:  # If the dictionary is empty, construct the entry from the path
                entry = {cat: val for cat, val in current_path}
                items.append(entry)
    return items

In [12]:
    # Define start and end timestamps
    start = '2022-07-18 00:00:00'
    end = '2022-10-13 21:00:00'
    
    # Generate the timestamp range at 1-hour intervals
    timestamp_range = pd.date_range(start=start, end=end, freq='h')
    
    # Create the DataFrame
    df_default = pd.DataFrame({
        'TIMESTAMP': timestamp_range,
        '#Events': 0,
        'Anomaly': 0
    })

In [13]:
df

,Call Type,Event Type,Subscriber Type,Traffic Type,Service Type,Partner Description,#Events,TIMESTAMP,Anomaly
0,SMS,ORIGINATED,Postpaid,OFF-NET,NOT-ROAMING,P_0001,105,2022-07-18 00:00:00,0
1,VOICE,CALL FORWARD,Application,INT,NOT-ROAMING,P_0001,16,2022-07-18 00:00:00,0
2,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0002,7,2022-07-18 00:00:00,0
3,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0003,15,2022-07-18 00:00:00,0
4,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0004,1,2022-07-18 00:00:00,0
...,...,...,...,...,...,...,...,...,...
958475,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0280,30,2022-10-13 21:00:00,0
958476,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0079,265,2022-10-13 21:00:00,0
958477,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0102,6,2022-10-13 21:00:00,0
958478,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0022,3,2022-10-13 21:00:00,0


In [14]:
df_default


,TIMESTAMP,#Events,Anomaly
0,2022-07-18 00:00:00,0,0
1,2022-07-18 01:00:00,0,0
2,2022-07-18 02:00:00,0,0
3,2022-07-18 03:00:00,0,0
4,2022-07-18 04:00:00,0,0
...,...,...,...
2105,2022-10-13 17:00:00,0,0
2106,2022-10-13 18:00:00,0,0
2107,2022-10-13 19:00:00,0,0
2108,2022-10-13 20:00:00,0,0


In [15]:
df["TIMESTAMP"] = pd.to_datetime(df['TIMESTAMP'])

In [16]:
df["TIMESTAMP"].dtype

dtype('<M8[ns]')

In [17]:
def set_leaf(df):
    df['Anomaly'] = df['Anomaly'].apply(lambda x: 1 if x > 0 else x)
    
    # Define start and end timestamps
    start = '2022-07-18 00:00:00'
    end = '2022-10-13 21:00:00'
    
    # Generate the timestamp range at 1-hour intervals
    timestamp_range = pd.date_range(start=start, end=end, freq='h')
    
    # Create the DataFrame
    df_default = pd.DataFrame({
        'TIMESTAMP': timestamp_range,
        '#Events': 0,
        'Anomaly': 0
    })
    print(df["TIMESTAMP"].dtype)
    print(df_default["TIMESTAMP"].dtype)
    
#     df_default.loc[df_default["TIMESTAMP"] == df["TIMESTAMP"], df.columns[1:]] = df.iloc[:, 1:].values
    # Merge original DataFrame with the default DataFrame based on 'TIMESTAMP'
    df_default = pd.merge(df_default, df, on='TIMESTAMP', how='left', suffixes=('', '_y'))
    
    # Overwrite columns with values from df if available
    for column in df.columns[1:]:  # assuming 'TIMESTAMP' is at index 0
        df_default[column] = df_default[column + '_y'].combine_first(df_default[column])
    
    # Drop helper columns
    df_default.drop(columns=[col for col in df_default.columns if '_y' in col], inplace=True)
    
    return df_default


In [18]:
def rm_empty_agg(df, ft_hierarchy):
    items = []
    # Wrap ft_hierarchy with tqdm for a progress bar
    print("Removing the empty aggregations from the hierarchy!")
    for conditions in tqdm(ft_hierarchy, desc='Processing conditions'):
        if contain_condition(filters, conditions):
            items.append(conditions)
    return items

In [19]:
def create_final_hierarchy(df, rm_ft_hierarchy):
    aggregated_df = {}
    
    # Wrap the enumeration with tqdm for better progress visibility
    for i, conditions in tqdm(enumerate(rm_ft_hierarchy, start=1), total=len(rm_ft_hierarchy), desc="Processing conditions"):
        agg_df = filter_and_aggregate(df, conditions, hierarchy=True)

        if not agg_df.empty:
            agg_df = set_leaf(agg_df)

        aggregated_df[f'leaf_{i}'] = agg_df

    return aggregated_df

In [20]:
def calculate_AUC(df):
    df_aux = df.copy()
    mean = df_aux["#Events"].mean()
    std = df_aux["#Events"].std()
    df_aux['z_score'] = (df_aux['#Events'] - mean) / std
    df_aux['AUC_score'] = norm.cdf(df_aux['z_score'])

    return df_aux['AUC_score'].mean()

In [21]:
def calculate_AUC_hierarchy(aggregated_df):
    auc_scores = {}

    # Using tqdm to show progress. Wrap the .items() of the dictionary with tqdm.
    for leaf_name, df_leaf in tqdm(aggregated_df.items(), desc="Calculating AUCs"):
        if df_leaf["Anomaly"].sum() > 0:
            auc_score = calculate_AUC(df_leaf)
            auc_scores[leaf_name] = auc_score

    # Calculate the mean AUC score across all DataFrames
    auc_score_mean = sum(auc_scores.values()) / len(auc_scores) if auc_scores else 0

    return auc_score_mean

In [22]:
def pipeline_hierarchy(df, n_hierarchies, available_categories):
    group_hierarchies = []
    auc_hierarchies = []
    rt_available_categories = revert_key_value_mapping(available_categories)
    for _ in range(n_hierarchies):
        hierarchy = create_hierarchy(available_categories)
        ft_hierarchy = flatten_dict_structure(hierarchy, rt_available_categories)
        rm_ft_hierarchy = rm_empty_agg(df, ft_hierarchy)
        aggregated_df = create_final_hierarchy(df, rm_ft_hierarchy)
        auc_hierarchy = calculate_AUC_hierarchy(aggregated_df)
        group_hierarchies.append(aggregated_df)
        auc_hierarchies.append(auc_hierarchy)
        
    return group_hierarchies, auc_hierarchies

In [23]:
df

,Call Type,Event Type,Subscriber Type,Traffic Type,Service Type,Partner Description,#Events,TIMESTAMP,Anomaly
0,SMS,ORIGINATED,Postpaid,OFF-NET,NOT-ROAMING,P_0001,105,2022-07-18 00:00:00,0
1,VOICE,CALL FORWARD,Application,INT,NOT-ROAMING,P_0001,16,2022-07-18 00:00:00,0
2,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0002,7,2022-07-18 00:00:00,0
3,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0003,15,2022-07-18 00:00:00,0
4,VOICE,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0004,1,2022-07-18 00:00:00,0
...,...,...,...,...,...,...,...,...,...
958475,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0280,30,2022-10-13 21:00:00,0
958476,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0079,265,2022-10-13 21:00:00,0
958477,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0102,6,2022-10-13 21:00:00,0
958478,SMS,TERMINATED,Roamer In,INT,NOT-ROAMING,P_0022,3,2022-10-13 21:00:00,0


In [24]:
group_hierarchies, auc_hierarchies = pipeline_hierarchy(df, 2, available_categories)

Removing the empty aggregations from the hierarchy!


Processing conditions:   7%|▋         | 2/27 [00:00<00:04,  5.25it/s]

datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


Processing conditions:  15%|█▍        | 4/27 [00:00<00:04,  5.71it/s]

datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


Processing conditions:  22%|██▏       | 6/27 [00:01<00:03,  5.87it/s]

... (1,846 outputs left out)

datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


Processing conditions: 100%|██████████| 932/932 [05:42<00:00,  2.72it/s]


datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


Calculating AUCs: 100%|██████████| 932/932 [00:01<00:00, 886.94it/s]


In [25]:
auc_hierarchies

[0.47615325597278724, 0.4625517768250284]

In [26]:
index = auc_hierarchies.index(max(auc_hierarchies))
best_result = group_hierarchies[index]

In [27]:
# Save to a pickle file
with open('dataframes_dict.pkl', 'wb') as handle:
    pickle.dump(best_result, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [28]:
# Load from a pickle file
# with open('dataframes_dict.pkl', 'rb') as handle:
#     loaded_dataframes_dict = pickle.load(handle)


# **5. Anomaly Detetion** <a class="anchor" id="5"></a>

[Table of Contents](#0.1)

1. label das anomalias
2. hierarquia fixa, calcular Matrix P
3. Optimizacao S

Como a Criacao de janelas series temporais relaciona-se com o seguintes:
Hierarquias
Detecao de anomalias
agoritmos genericos

Matrix Profile
Mais tarde: Pyod (outlier detection)

Distancia minima
AUC (‘threshold’)

## Ao aumentar o quartile a anomalia restringe aos valores dos eventos com duracao excessiva, apesar da duranca aumentar com o tempo seguindo uma periodicidade semanal

verificar as anomalias serem iguais entre os dois colunas ja que tem o mesmo numero de anomalias do Event Duration and #Events

## 5.1 Statistical Thresholding

In [29]:
# def create_features(data, window_size):
#     X = []
#     y = []
#     for i in range(len(data) - window_size):
#         X.append(data.iloc[i:i+window_size].drop(columns=['Anomaly']).values.flatten())
#         y.append(data.iloc[i+window_size]['Anomaly'])
#     return np.array(X), np.array(y)


# def plot_confusion_matrix(y_true, y_pred, labels=None, title='Confusion Matrix'):
#     cm = confusion_matrix(y_true, y_pred, labels=labels)
#     disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
#     disp.plot(cmap=plt.cm.Blues)
#     plt.title(title)
#     plt.show()
    
    
# def sliding_window_auc(window_size, df, plot=False):
#     # Convert TIMESTAMP to datetime and sort
#     data = df.set_index("TIMESTAMP")


#     # Split into train and test sets
#     X, y = create_features(data, window_size)
    
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
#     # Train the Logistic Regression model
#     model = LogisticRegression(max_iter=1000)
#     print(X_train.size, y_train.size)
#     model.fit(X_train, y_train)

#     # Get predicted probabilities for the positive class
#     probabilities = model.predict_proba(X_test)[:, 1]

#     # Compute AUC
#     auc_score = roc_auc_score(y_test, probabilities)

#     if plot:
#         # Use the predictions, not the probabilities, to plot the confusion matrix
#         y_pred = model.predict(X_test)
#         plot_confusion_matrix(y_test, y_pred)
    
#     return auc_score

In [30]:
# def HTS_auc(aggregated_df, window_size):
#     auc_scores = {}

#     # Using tqdm to show progress. Wrap the .items() of the dictionary with tqdm.
#     for leaf_name, df_leaf in tqdm(aggregated_df.items(), desc="Calculating AUCs"):
#         if df_leaf["Anomaly"].sum() > 0:
#             auc_score = sliding_window_auc(window_size, df=df_leaf, plot=False)
#             auc_scores[leaf_name] = auc_score

#     # Calculate the mean AUC score across all DataFrames
#     auc_score_mean = sum(auc_scores.values()) / len(auc_scores) if auc_scores else 0

#     return auc_score_mean

# HTS_auc(aggregated_df, 24)

As hierarquias so entram aqui nesta fase?
as anomalias agregam, so basta ter um dos valores da juncao positiva para ser anomalia

Amostragem aleatória das hierarquias
Definir população inicial
Como fazer as mutações

Genetic Programing

Ex: 
 > Comocar com 100 aleatorias e avaliamos o fitness (F1 score)
 > > 100 hieraquias aleatorias
 > Escolher as 20 melhores e fazer multacoes ate termos uma nova populacao de 100 
 > Paragem:
 > > O melhor F1 score da populacao nao aumenta mais que 1%
 > >
